### Connect to Ollama

In [35]:
import ollama

### Connect to Qdrant


In [32]:
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")

### Add new collection

In [ ]:
from qdrant_client.models import Distance, VectorParams

client.create_collection(
    collection_name="ollama_test_collection",
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)

### Process pdf file


In [22]:
import pypdf
import os

data_path = '../data/data/INFORMATION-TECHNOLOGY'

pdf_files = [f for f in os.listdir(data_path) if f.endswith('.pdf')]
pdf_files.sort()
pdf_files = pdf_files[:5]

pdf_readers = [pypdf.PdfReader(os.path.join(data_path, pdf_file)) for pdf_file in pdf_files]

all_texts = []
for pdf_reader in pdf_readers:
    pdf_text = ''
    for page in pdf_reader.pages:
        extracted = page.extract_text()
        if extracted:
            pdf_text += extracted
    all_texts.append(pdf_text)

for idx, doc_text in enumerate(all_texts):
    print(f"PDF {idx+1} length: {len(doc_text)}")
    print(f"PDF {idx+1} preview: {doc_text[:100]}")


PDF 1 length: 8318
PDF 1 preview: INFORMATION TECHNOLOGY TECHNICIAN I
Summary
Versatile Systems Administrator possessing superior trou
PDF 2 length: 7140
PDF 2 preview: INFORMATION TECHNOLOGY MANAGER
Professional Summary
Possesses an extensive background in Information
PDF 3 length: 5192
PDF 3 preview: WORKING RF SYSTEMS ENGINEER
Qualifications
Microsoft office/Office for Mac, pages, numbers, keynote 
PDF 4 length: 5115
PDF 4 preview: INFORMATION TECHNOLOGY MANAGER
Summary
Dedicated 
IT Manager
 
well-versed in analyzing and mitigati
PDF 5 length: 5377
PDF 5 preview: IT MANAGEMENT
Career Overview
Detail-oriented professional with extensive Information Technology exp


### Create a collection for each CV

In [ ]:
for idx, doc_text in enumerate(all_texts):
    client.create_collection(
        collection_name=f"ollama_test_collection_{idx}",
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
    )

### Divide all texts into chunks of 512 characters

In [29]:
for idx, doc_text in enumerate(all_texts):
    chunks = [doc_text[i:i+512] for i in range(0, len(doc_text), 512)]
    for chunk_idx, chunk in enumerate(chunks):
        embedding = ollama.embeddings(
            model='mxbai-embed-large',
            prompt=chunk,
        )['embedding']

        client.upsert(
            collection_name=f"ollama_test_collection_{idx}",
            wait=True,
            points=[{
                "id": idx * 100000 + chunk_idx,
                "vector": embedding,
                "payload": {"text": chunk},
            }],
        )

### Find best CV matches for each job posting

In [40]:
job_postings_path = '../data/job_postings'

job_postings = {}
for filename in sorted(os.listdir(job_postings_path)):
    if filename.endswith('.txt'):
        file_path = os.path.join(job_postings_path, filename)
        with open(file_path, 'r', encoding='utf-8') as file:
            job_postings[filename] = file.read()

for posting_name, posting_text in job_postings.items():
    posting_embedding = ollama.embeddings(
        model='mxbai-embed-large',
        prompt=posting_text,
    )['embedding']

    best_match = None

    for idx in range(len(all_texts)):
        collection_name = f"ollama_test_collection_{idx}"

        point_count = client.count(
            collection_name=collection_name,
            exact=True,
        ).count

        if point_count == 0:
            continue

        result = client.query_points(
            collection_name=collection_name,
            query=posting_embedding,
            with_payload=True,
            limit=point_count,
        ).points

        if result:
            aggregated_score = sum(point.score for point in result) / len(result)
            top_point = result[0]

            if best_match is None or aggregated_score > best_match['score']:
                best_match = {
                    'collection_name': collection_name,
                    'cv_index': idx,
                    'score': aggregated_score,
                    'point_count': point_count,
                    'chunk_text': top_point.payload.get('text', '') if top_point.payload else '',
                }

    print(f"Job posting: {posting_name}")
    if best_match is None:
        print("  No matches found.")
    else:
        print(f"  Best CV: {best_match['collection_name']} (CV #{best_match['cv_index'] + 1})")
        print(f"  Aggregated similarity score (mean across all {best_match['point_count']} points): {best_match['score']:.4f}")
        print(f"  Top matching chunk preview: {best_match['chunk_text'][:220]}")
    print('-' * 80)

Job posting: cloud_systems_administrator.txt
  Best CV: ollama_test_collection_0 (CV #1)
  Aggregated similarity score (mean across all 17 points): 0.6106
  Top matching chunk preview: INFORMATION TECHNOLOGY TECHNICIAN I
Summary
Versatile Systems Administrator possessing superior troubleshooting skills for networking issues, end user problems, and network security.
Experienced in server management, sys
--------------------------------------------------------------------------------
Job posting: it_manager_enterprise_ops.txt
  Best CV: ollama_test_collection_3 (CV #4)
  Aggregated similarity score (mean across all 10 points): 0.6140
  Top matching chunk preview: INFORMATION TECHNOLOGY MANAGER
Summary
Dedicated 
IT Manager
 
well-versed in analyzing and mitigating risk and finding cost-effective solutions. Excels at boosting performance and
productivity by establishing realistic 
--------------------------------------------------------------------------------
Job posting: rf_systems_engi